In [1]:
#CRIAÇÃO DA TABELA SILVER INVENTORY MOVEMENTS
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))

# ============================================================
# CRIAÇÃO DA TABELA SILVER INVENTORY MOVEMENTS
# ============================================================
#
# Objetivo:
# Criar a tabela silver__inventory_movements__lite a partir da
# camada Bronze, aplicando regras de limpeza e padronização.
#
# Transformações realizadas:
# - Remoção de espaços extras (TRIM)
# - Padronização para letras maiúsculas (UPPER)
# - Conversão segura da coluna de data
# - Preservação das demais colunas necessárias para análise
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE silver__inventory_movements__lite AS

-- ============================================================
-- CTE responsável pela limpeza inicial dos dados
-- ============================================================
WITH base AS (

    SELECT

        -- Remove espaços extras e converte o ID para maiúsculo
        UPPER(TRIM(movement_id)) AS movement_id,

        -- Mantém a data original para tratamento posterior
        movement_date,

        -- Padroniza o tipo de movimentação
        UPPER(TRIM(movement_type)) AS movement_type,

        -- Padroniza o SKU do produto
        UPPER(TRIM(sku)) AS sku,

        -- Padroniza a localização de origem
        UPPER(TRIM(from_location_id)) AS from_location_id,

        -- Padroniza a localização de destino
        UPPER(TRIM(to_location_id)) AS to_location_id,

        -- Quantidade movimentada
        qty

    FROM bronze__inventory_movements__lite
)

-- ============================================================
-- Seleção final dos dados tratados
-- ============================================================
SELECT

    -- Identificador da movimentação
    movement_id,

    -- Conversão segura da data:
    -- 1º tenta converter diretamente para DATE
    -- 2º tenta interpretar o formato YYYY-MM-DD
    -- Se ambas falharem, retorna NULL
    COALESCE(
        TRY_CAST(movement_date AS DATE),
        TRY_STRPTIME(movement_date, '%Y-%m-%d')::DATE
    ) AS movement_date,

    -- Tipo da movimentação
    movement_type,

    -- SKU do produto
    sku,

    -- Local de origem
    from_location_id,

    -- Local de destino
    to_location_id,

    -- Quantidade movimentada
    qty

FROM base;
""")
# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "silver__inventory_movements__lite" in tables["name"].values:
    print("✅ Tabela silver__inventory_movements__lite criada com sucesso!")
else:
    print("❌ Tabela silver__inventory_movements__lite não foi criada!")
conn.close()

✅ Tabela silver__inventory_movements__lite criada com sucesso!
